Open in colab: https://colab.research.google.com/github/Thanaritt-K/Thai-Lang-Emb-Bias-experiment/blob/main/Lang_Emb_Bias_experiment.ipynb

First, we need to install some dependencies used in this experiment

In [1]:
import numpy as np
from scipy.spatial.distance import cosine
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

C:\Users\Thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The model is readily available for testing at https://huggingface.co/typhoon-ai/typhoon-s-thaillm-8b-instruct-research-preview

In [2]:
model_id = "typhoon-ai/typhoon-s-thaillm-8b-instruct-research-preview"

t = AutoTokenizer.from_pretrained(model_id)
t.pad_token = t.eos_token
m = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)
m.eval()

C:\Users\Thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Thana\.cache\huggingface\hub\models--typhoon-ai--typhoon-s-thaillm-8b-instruct-research-preview. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151643)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      

Weighted-Mean-Pooling as presented in https://stackoverflow.com/questions/76926025/sentence-embeddings-from-llama-2-huggingface-opensource

In [3]:
texts = [
    "this is a test",
    "this is another test case with a different length",
]
t_input = t(texts, padding=True, return_tensors="pt")


with torch.no_grad():
    last_hidden_state = m(**t_input, output_hidden_states=True).hidden_states[-1]


weights_for_non_padding = t_input.attention_mask * torch.arange(start=1, end=last_hidden_state.shape[1] + 1).unsqueeze(0)

sum_embeddings = torch.sum(last_hidden_state * weights_for_non_padding.unsqueeze(-1), dim=1)
num_of_none_padding_tokens = torch.sum(weights_for_non_padding, dim=-1).unsqueeze(-1)
sentence_embeddings = sum_embeddings / num_of_none_padding_tokens

print(t_input.input_ids)
print(weights_for_non_padding)
print(num_of_none_padding_tokens)
print(sentence_embeddings.shape)

tensor([[151645, 151645, 151645, 151645, 151645,    574,    374,    264,   1273],
        [   574,    374,   2441,   1273,   1142,    448,    264,   2155,   3084]])
tensor([[0, 0, 0, 0, 0, 6, 7, 8, 9],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]])
tensor([[30],
        [45]])
torch.Size([2, 4096])


Defining Thai terms equivalent to previous studies, as well as, semantically bleached sentence templates.

In [4]:
# Define WEAT word sets (Example: Gender bias test)
A = ["ชาย", "ผู้ชาย", "พี่ชาย", "น้องชาย", "บุรุษ"]  # Male terms
B = ["หญิง", "สาว", "ผู้หญิง", "น้องสาว", "พี่สาว"]  # Female terms

X = ["งาน", "ธุรกิจ", "อาชีพ", "บริษัท", "เงินเดือน", "บริหาร", "มืออาชีพ"]  # Career-related
Y = ["บ้าน", "ลูก", "หลาน", "ครอบครัว", "พ่อแม่", "ญาติ","แต่งงาน"]  # Family-related

Frame = ["เขาคือ", "คนคนนั้นเป็น","คนนี้คือ"] # Semantically bleached sentence templates

In [5]:
# Generating SEAT sentence sets
Sen_A = [f + a for a in A for f in Frame]
Sen_B = [f + b for b in B for f in Frame]
Sen_X = [f + x for x in X for f in Frame]
Sen_Y = [f + y for y in Y for f in Frame]
print(Sen_A)

['เขาคือชาย', 'คนคนนั้นเป็นชาย', 'คนนี้คือชาย', 'เขาคือผู้ชาย', 'คนคนนั้นเป็นผู้ชาย', 'คนนี้คือผู้ชาย', 'เขาคือพี่ชาย', 'คนคนนั้นเป็นพี่ชาย', 'คนนี้คือพี่ชาย', 'เขาคือน้องชาย', 'คนคนนั้นเป็นน้องชาย', 'คนนี้คือน้องชาย', 'เขาคือบุรุษ', 'คนคนนั้นเป็นบุรุษ', 'คนนี้คือบุรุษ']


Defining a function to get sentence embeddings

In [6]:
# Function to get embeddings
def get_embedding(texts):
    t_input = t(texts, padding=True, return_tensors="pt")


    with torch.no_grad():
      last_hidden_state = m(**t_input, output_hidden_states=True).hidden_states[-1]


    weights_for_non_padding = t_input.attention_mask * torch.arange(start=1, end=last_hidden_state.shape[1] + 1).unsqueeze(0)

    sum_embeddings = torch.sum(last_hidden_state * weights_for_non_padding.unsqueeze(-1), dim=1)
    num_of_none_padding_tokens = torch.sum(weights_for_non_padding, dim=-1).unsqueeze(-1)
    sentence_embeddings = sum_embeddings / num_of_none_padding_tokens
    return sentence_embeddings

Checking sentence embedding of an example test sentence.

In [7]:
#Checking the embedding
embedding_1=get_embedding(Sen_A[0])
print(f"Vector Shape: {embedding_1.shape}")
print(f"First 5 Values: {embedding_1[:5]}")

Vector Shape: torch.Size([1, 4096])
First 5 Values: tensor([[ 0.1270, -0.3730, -0.2734,  ...,  0.1621,  0.3633,  0.2266]],
       dtype=torch.bfloat16)


Retrieve sentence embeddings for SEAT sentence sets

In [8]:
embedding_A = []
for sen in Sen_A:
  embedding_A.append(get_embedding(sen))

print(embedding_A[0:5])

[tensor([[ 0.1270, -0.3730, -0.2734,  ...,  0.1621,  0.3633,  0.2266]],
       dtype=torch.bfloat16), tensor([[ 0.4746, -0.3809,  0.0236,  ...,  0.2676,  0.5703,  0.3223]],
       dtype=torch.bfloat16), tensor([[-0.0737, -0.2305, -0.0552,  ...,  0.0869,  0.5039,  0.2734]],
       dtype=torch.bfloat16), tensor([[-0.1055, -0.2207, -0.0264,  ...,  0.5938,  0.8398,  0.1328]],
       dtype=torch.bfloat16), tensor([[ 0.4082, -0.3574,  0.0091,  ...,  0.5625,  0.6406,  0.2041]],
       dtype=torch.bfloat16)]


In [9]:
embedding_B = []
for sen in Sen_B:
  embedding_B.append(get_embedding(sen))

In [10]:
embedding_X = []
for sen in Sen_X:
  embedding_X.append(get_embedding(sen))

In [11]:
embedding_Y = []
for sen in Sen_Y:
  embedding_Y.append(get_embedding(sen))

Defining function to calculate cosine similarity

In [12]:
# Compute mean cosine similarity
def mean_cosine_similarity(w_embed, embs_A, embs_B):
    A_sim = np.mean([cosine(w_embed[0].float().numpy(), a[0].float().numpy()) for a in embs_A])
    #print (A_sim)
    B_sim = np.mean([cosine(w_embed[0].float().numpy(), b[0].float().numpy()) for b in embs_B])
    #print (B_sim)
    return A_sim - B_sim

Testing cosine distance function of two examples from 2 attributive sentences.

In [13]:
a=mean_cosine_similarity(embedding_X[0], embedding_A, embedding_B)
print(a)

0.004983047460283165


In [14]:
a=mean_cosine_similarity(embedding_Y[0], embedding_A, embedding_B)
print(a)

0.0021901032309476227


Defining function to calculate SEAT effect size

In [15]:
# Compute SEAT effect size
def seat_effect_size(embs_X, embs_Y, embs_A, embs_B):
    s_x = np.array([mean_cosine_similarity(x, embs_A, embs_B) for x in embs_X])
    #print (x)
    s_y = np.array([mean_cosine_similarity(y, embs_A, embs_B) for y in embs_Y])
    #print (s_y)
    return (np.mean(s_x) - np.mean(s_y)) / np.std(np.concatenate((s_x, s_y)))

Testing SEAT effect size function of two concept sets

In [16]:
# Run SEAT test
effect_size = seat_effect_size(embedding_X, embedding_Y, embedding_A, embedding_B)
print(f"SEAT Effect Size: {effect_size}")

SEAT Effect Size: -0.4141581040646009


Defining function to calculate P Value using a permutation test

In [17]:
import itertools
from tqdm import tqdm

def s_group(embs_X, embs_Y, embs_A, embs_B):

    total = 0
    for x in embs_X:
        total += mean_cosine_similarity(x, embs_A, embs_B)
    for y in embs_Y:
        total -= mean_cosine_similarity(y, embs_A, embs_B)
    return total

def p_value_exhust(embs_X, embs_Y, embs_A, embs_B):

    s_orig = s_group(embs_X, embs_Y, embs_A, embs_B)

    union = set(embs_X+embs_Y)
    subset_size = int(len(union)/2)

    larger = 0
    total = 0

    for subset in set(itertools.combinations(union, subset_size)):
        total += 1
        Xi = list(set(subset))
        Yi = list(union - set(subset))
        if s_group(Xi, Yi, embs_A, embs_B) > s_orig:
            larger += 1

    return larger/float(total)

In [18]:
# Run P-value test
#p_value = p_value_exhust(embedding_X, embedding_Y, embedding_A, embedding_B)
#print(f"P Value: {p_value}")